# 05 - Weights + Meta-dataset

Loops `resolve_pipeline_seeds()` (override with `TM_RUN_SEEDS`). For each seed, rebuilds prompts from that seed's probabilities into `artifacts/seeds/{seed}/`.

1. Loads `val_probs_*.npy` for every base model and computes per-model validation accuracy.
2. Computes accuracy-normalized weights `w_i = acc_i / sum_j acc_j` and saves them to `weights.json`.
3. Renders structured-token prompts for `train` (using OOF probs), `val`, and `test` and writes JSONL to `meta_jsonl/{train,val,test}.jsonl`.

Train rows include the `<label>...</label>` completion (gold). Val rows include the gold completion too (for SFT eval). Test rows store the gold label in a separate `gold` field so the LLM never sees it during generation.

In [1]:
import sys, os, json

REPO_ROOT = '/content/drive/MyDrive/thesis/topicmodeling'
TMP_ROOT = '/content/ensemble_tmp'

if 'google.colab' in sys.modules:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
    if REPO_ROOT not in sys.path:
        sys.path.insert(0, REPO_ROOT)
else:
    import pathlib
    LOCAL = pathlib.Path.cwd().resolve()
    while LOCAL.name != 'tm_research' and LOCAL.parent != LOCAL:
        LOCAL = LOCAL.parent
    sys.path.insert(0, str(LOCAL.parent))

from tm_research.ensemble.colab_setup import setup_colab
paths = setup_colab(repo_root=REPO_ROOT, tmp_root=TMP_ROOT)

Mounted at /content/drive
[utils_io] pulled 234 files: /content/drive/MyDrive/thesis/topicmodeling/tm_research/ensemble/artifacts -> /content/ensemble_tmp/artifacts
[colab_setup] is_colab=True
[colab_setup] repo_root=/content/drive/MyDrive/thesis/topicmodeling
[colab_setup] tmp_root=/content/ensemble_tmp (hf_cache/bert_work/lora_work)
[colab_setup] artifacts=/content/ensemble_tmp/artifacts
[colab_setup] persistent_artifacts=/content/drive/MyDrive/thesis/topicmodeling/tm_research/ensemble/artifacts
[colab_setup] HF_TOKEN: set


In [2]:
import numpy as np
from tm_research.ensemble import utils_io as uio
from tm_research.ensemble.utils_io import (
    load_splits, load_probs, activate_seed_artifacts, reset_artifacts_root,
)
from tm_research.ensemble.multiseed import resolve_pipeline_seeds, seed_is_complete
from tm_research.ensemble.utils_stacking import (
    BASE_MODEL_NAMES, compute_weights, format_prompt, write_jsonl, weighted_average,
    build_meta_system_prompt, PROMPT_SCHEMA_VERSION,
)

train_df, val_df, test_df, label_map = load_splits()
y_train = train_df['label'].map(label_map.label2id).to_numpy()
y_val = val_df['label'].map(label_map.label2id).to_numpy()
y_test = test_df['label'].map(label_map.label2id).to_numpy()
print('classes', label_map.class_names)
print('shapes', len(train_df), len(val_df), len(test_df))

classes ['Anger', 'Disgust', 'Enjoyment', 'Fear', 'Other', 'Sadness', 'Surprise']
shapes 10549 686 692


In [3]:
REQUIRED_RELATIVE_PATHS = (
    'weights.json',
    'meta_jsonl/train.jsonl',
    'meta_jsonl/val.jsonl',
    'meta_jsonl/test.jsonl',
)

def build_rows(texts, probs_per_model, gold_labels=None, with_completion=True):
    rows = []
    n = len(texts)
    for i in range(n):
        per_model = {m: probs_per_model[m][i] for m in BASE_MODEL_NAMES}
        gold = gold_labels[i] if gold_labels is not None else None
        formatted = format_prompt(
            text=texts[i],
            probs_per_model=per_model,
            weights=weights,
            label_map=label_map,
            label=gold if with_completion else None,
        )
        row = {
            'idx': i,
            'text': texts[i],
            'prompt': formatted['prompt'],
            'completion': formatted['completion'],
        }
        if gold is not None:
            row['gold'] = gold
        rows.append(row)
    return rows

## Per-seed weights and JSONL

In [4]:
for seed in resolve_pipeline_seeds():
    if seed_is_complete(seed, REQUIRED_RELATIVE_PATHS):
        print(f'skip seed {seed}')
        continue
    activate_seed_artifacts(seed)
    try:
        oof_probs = {m: load_probs(m, 'oof') for m in BASE_MODEL_NAMES}
        val_probs = {m: load_probs(m, 'val') for m in BASE_MODEL_NAMES}
        test_probs = {m: load_probs(m, 'test') for m in BASE_MODEL_NAMES}
        for m in BASE_MODEL_NAMES:
            print(f'{m:8s}  oof={oof_probs[m].shape}  val={val_probs[m].shape}  test={test_probs[m].shape}')

        val_acc = {m: float((val_probs[m].argmax(1) == y_val).mean()) for m in BASE_MODEL_NAMES}
        weights = compute_weights(val_acc)
        print('val accuracies:')
        for m, a in val_acc.items():
            print(f'  {m:8s}  acc={a:.4f}  w={weights[m]:.4f}')

        weighted_val = weighted_average(val_probs, weights)
        weighted_val_acc = float((weighted_val.argmax(1) == y_val).mean())
        print(f'weighted-avg val accuracy: {weighted_val_acc:.4f}')

        system_prompt = build_meta_system_prompt(label_map, weights)
        print('\nSystem prompt preview:\n', system_prompt)

        with open(uio.ARTIFACTS_DIR / 'weights.json', 'w', encoding='utf-8') as f:
            json.dump({
                'method': 'accuracy_normalized',
                'val_accuracy_per_model': val_acc,
                'weights': weights,
                'weighted_val_accuracy': weighted_val_acc,
                'prompt_schema': PROMPT_SCHEMA_VERSION,
                'system_prompt': system_prompt,
            }, f, ensure_ascii=False, indent=2)
        print('saved', uio.ARTIFACTS_DIR / 'weights.json')

        train_rows = build_rows(
            train_df['text'].tolist(), oof_probs,
            gold_labels=train_df['label'].tolist(), with_completion=True,
        )
        val_rows = build_rows(
            val_df['text'].tolist(), val_probs,
            gold_labels=val_df['label'].tolist(), with_completion=True,
        )
        test_rows = build_rows(
            test_df['text'].tolist(), test_probs,
            gold_labels=test_df['label'].tolist(), with_completion=False,
        )

        write_jsonl(train_rows, uio.META_JSONL_DIR / 'train.jsonl')
        write_jsonl(val_rows, uio.META_JSONL_DIR / 'val.jsonl')
        write_jsonl(test_rows, uio.META_JSONL_DIR / 'test.jsonl')
        print('wrote', len(train_rows), 'train,', len(val_rows), 'val,', len(test_rows), 'test')
        print('first train prompt:\n', train_rows[0]['prompt'])
        print('completion:', train_rows[0]['completion'])
    finally:
        reset_artifacts_root()

phobert   oof=(10549, 7)  val=(686, 7)  test=(692, 7)
cafebert  oof=(10549, 7)  val=(686, 7)  test=(692, 7)
vibert    oof=(10549, 7)  val=(686, 7)  test=(692, 7)
logreg    oof=(10549, 7)  val=(686, 7)  test=(692, 7)
svc       oof=(10549, 7)  val=(686, 7)  test=(692, 7)
val accuracies:
  phobert   acc=0.6210  w=0.2067
  cafebert  acc=0.6501  w=0.2164
  vibert    acc=0.5481  w=0.1824
  logreg    acc=0.5933  w=0.1975
  svc       acc=0.5918  w=0.1970
weighted-avg val accuracy: 0.6676

System prompt preview:
 You are an emotion classifier for Vietnamese social-media text. Pick exactly one label from: Anger, Disgust, Enjoyment, Fear, Other, Sadness, Surprise. You receive: [TEXT] the input sentence [/TEXT], then [WEIGHTED_AVG] the weighted-average probability over all base models [/WEIGHTED_AVG], then [STACK] per-model probabilities [/STACK]. Label abbreviations: A=Anger, D=Disgust, J=Enjoyment, F=Fear, O=Other, Sa=Sadness, Su=Surprise. Model abbreviations: p=phobert, c=cafebert, v=vibert, l=

JSONL files are written inside the seed loop above (`artifacts/seeds/{seed}/meta_jsonl/`).

In [5]:
print('Per-seed weights.json and meta_jsonl live under artifacts/seeds/{seed}/')

Per-seed weights.json and meta_jsonl live under artifacts/seeds/{seed}/


In [6]:
from tm_research.ensemble.utils_io import push_artifacts_to_persistent
push_artifacts_to_persistent()

[utils_io] pushed 246 files: /content/ensemble_tmp/artifacts -> /content/drive/MyDrive/thesis/topicmodeling/tm_research/ensemble/artifacts


PosixPath('/content/drive/MyDrive/thesis/topicmodeling/tm_research/ensemble/artifacts')

In [7]:
from google.colab import runtime
runtime.unassign()